In [1]:
%%sql
CREATE OR REPLACE TABLE silver_budget USING DELTA AS
SELECT 
    date,
    site,
    department,
    AVG(budget_eur) as monthly_budget_eur,
    AVG(actual_eur) as monthly_actual_eur,
    
    -- STORM FOUNDATION (you have)
    AVG(actual_eur/budget_eur) as variance_ratio,
    ROUND(AVG((actual_eur - budget_eur)/budget_eur * 100), 2) as variance_pct,
    
    -- NEW: PRODUCTION FINANCE KPIs
    ROUND(AVG(actual_eur - budget_eur), 0) as budget_variance_eur,
    CASE 
        WHEN AVG(actual_eur/budget_eur) > 1.05 THEN 'Over'
        WHEN AVG(actual_eur/budget_eur) < 0.95 THEN 'Under' 
        ELSE 'OnTrack'
    END as budget_status,
    
    -- PROFITABILITY (Solver CPM core)
    CASE WHEN category = 'revenue' THEN AVG(actual_eur) ELSE 0 END as revenue_eur,
    CASE WHEN category = 'expense' THEN AVG(actual_eur) ELSE 0 END as expense_eur,
    GREATEST(0, SUM(CASE WHEN category = 'revenue' THEN actual_eur ELSE 0 END) - 
                 SUM(CASE WHEN category = 'expense' THEN actual_eur ELSE 0 END)) as gross_profit_eur,
    
    -- EFFICIENCY RATIOS (CFO dashboards)
    AVG(actual_eur) / NULLIF(GREATEST(1, AVG(budget_eur)), 0) as spend_efficiency,
    
    -- SITE PERFORMANCE (Irish manufacturing)
    NTILE(4) OVER (PARTITION BY date ORDER BY AVG(actual_eur) DESC) as site_performance_quartile,
    
    COUNT(*) as record_count
FROM bronze_erp 
GROUP BY date, site, department, category
ORDER BY date DESC, variance_pct;

-- Verify enhanced KPIs
SELECT * FROM silver_budget LIMIT 5


StatementMeta(, e3a90eae-ca12-49b4-9763-280fa5426984, 3, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 5 rows and 15 fields>